<a href="https://colab.research.google.com/github/anispee/model-1/blob/main/amazon_dataset_2025_test_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cp '/content/drive/MyDrive/amazon 2025.zip'

In [ ]:
!unzip -q '/content/amazon 2025.zip' -d '/content/amazon_2025_unzipped'

The `amazon 2025.zip` file has been unzipped into the `/content/amazon_2025_unzipped` directory. Let's list the contents of this new directory to see what's inside.

In [ ]:
!ls -F '/content/amazon_2025_unzipped/'

In [ ]:
print(df.columns)

## Stage 1: Data Cleaning (Full Dataset)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Download necessary NLTK data if not already present
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')


In [ ]:
# 1. Handle missing values in the 'Product' column by filling with an empty string
print(f"Missing values in 'Product' before filling: {df['Product'].isnull().sum()}")
df['Product'] = df['Product'].fillna('')
print(f"Missing values in 'Product' after filling: {df['Product'].isnull().sum()}")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and special characters (keep alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Convert to lowercase and strip whitespace
    text = text.lower().strip()
    return text

# Apply cleaning to the 'Product' column and create a new 'Cleaned_Product' column
df['Cleaned_Product'] = df['Product'].apply(clean_text)

print("\nOriginal Product (sample):\n", df['Product'].head().to_string())
print("\nCleaned Product (sample):\n", df['Cleaned_Product'].head().to_string())


## Stage 2: Data Splitting (Train / Validation / Test)

In [ ]:
# Split data into training and temporary sets (80% train, 20% temp)
# shuffle=False is used to maintain the original order of the data for time-series-like splitting.
# random_state and stratify are removed as they are typically used with shuffling.
train_df, temp_df = train_test_split(df, test_size=0.2, shuffle=False)

# Split temporary set into validation and test sets (50% validation, 50% test from temp)
val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=False)

print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")


## Stage 3: Tokenization, Stopword Removal, and TF-IDF Vectorization (Per Split)

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_tokens(text):
    tokens = word_tokenize(text)
    # Remove stopwords and join back into a string
    return ' '.join([w for w in tokens if w not in stop_words])

# Apply tokenization and stopword removal to each split
train_df['Processed_Text'] = train_df['Cleaned_Product'].apply(preprocess_tokens)
val_df['Processed_Text'] = val_df['Cleaned_Product'].apply(preprocess_tokens)
test_df['Processed_Text'] = test_df['Cleaned_Product'].apply(preprocess_tokens)

print("Processed Text (Train sample):\n", train_df['Processed_Text'].head().to_string())
print("\nProcessed Text (Validation sample):\n", val_df['Processed_Text'].head().to_string())
print("\nProcessed Text (Test sample):\n", test_df['Processed_Text'].head().to_string())


In [ ]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000) # You can adjust max_features as needed

# Fit on training data and transform training data
train_tfidf = tfidf_vectorizer.fit_transform(train_df['Processed_Text'])

# Transform validation and test data using the fitted vectorizer
val_tfidf = tfidf_vectorizer.transform(val_df['Processed_Text'])
test_tfidf = tfidf_vectorizer.transform(test_df['Processed_Text'])

print(f"TF-IDF Matrix shape for Training Data: {train_tfidf.shape}")
print(f"TF-IDF Matrix shape for Validation Data: {val_tfidf.shape}")
print(f"TF-IDF Matrix shape for Test Data: {test_tfidf.shape}")

print("\nSample feature names from TF-IDF vectorizer:")
print(tfidf_vectorizer.get_feature_names_out()[:10])


## Stage 4: Data Exploration and Feature Engineering (Per Split)

### 4.1 Date Feature Extraction

In [ ]:
# Install holidays library if not already installed
!pip install holidays

import holidays
import pandas as pd

# Define the country for holidays (e.g., 'US' for United States, 'KR' for South Korea)
# You might need to adjust this based on your data's region
country_holidays = holidays.CountryHoliday('KR', years=range(2025, 2026)) # Assuming data is for 2025

In [ ]:
def extract_date_features(df_input):
    df_output = df_input.copy()
    # Ensure 'Date' column is datetime type
    df_output['Date'] = pd.to_datetime(df_output['Date'])

    df_output['year'] = df_output['Date'].dt.year
    df_output['month'] = df_output['Date'].dt.month
    df_output['day'] = df_output['Date'].dt.day
    df_output['day_of_week'] = df_output['Date'].dt.dayofweek # Monday=0, Sunday=6
    df_output['week_of_year'] = df_output['Date'].dt.isocalendar().week.astype(int)
    df_output['is_weekend'] = df_output['day_of_week'].isin([5, 6]).astype(int)

    # Check for holidays
    df_output['is_holiday'] = df_output['Date'].apply(lambda x: x in country_holidays).astype(int)

    return df_output

# Apply date feature extraction to each dataframe
train_df = extract_date_features(train_df)
val_df = extract_date_features(val_df)
test_df = extract_date_features(test_df)

print("Train DataFrame with new date features (head):\n")
display(train_df[['Date', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend', 'is_holiday']].head())

print("\nValidation DataFrame with new date features (head):\n")
display(val_df[['Date', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend', 'is_holiday']].head())

print("\nTest DataFrame with new date features (head):\n")
display(test_df[['Date', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend', 'is_holiday']].head())


### 4.2 Categorical Data Transformation

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Category', 'Payment Method', 'Customer Location']

# Initialize LabelEncoders for each categorical column
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Fit encoder only on the training data to avoid data leakage
    train_df[col + '_encoded'] = le.fit_transform(train_df[col])

    # Transform validation and test data using the fitted encoder
    # Handle new categories that might appear in val/test but not in train
    # Assign -1 for unknown categories in validation/test sets
    val_df[col + '_encoded'] = val_df[col].map(lambda s: -1 if s not in le.classes_ else le.transform([s])[0])
    test_df[col + '_encoded'] = test_df[col].map(lambda s: -1 if s not in le.classes_ else le.transform([s])[0])

    label_encoders[col] = le

print("Train DataFrame with encoded categorical features (head):\n")
display(train_df[categorical_cols + [col + '_encoded' for col in categorical_cols]].head())

print("\nValidation DataFrame with encoded categorical features (head):\n")
display(val_df[categorical_cols + [col + '_encoded' for col in categorical_cols]].head())

print("\nTest DataFrame with encoded categorical features (head):\n")
display(test_df[categorical_cols + [col + '_encoded' for col in categorical_cols]].head())


### 4.3 Sliding Window (Lag) Features

In [ ]:
def add_weekly_lag_features(df_split):
    df_temp = df_split.copy()

    # Ensure 'Date' column is datetime type, as it might be lost during previous operations or might not be sorted
    df_temp['Date'] = pd.to_datetime(df_temp['Date'])

    # Sort data for meaningful lag calculations within groups
    df_temp = df_temp.sort_values(by=['Product', 'Date'])

    # Calculate Last Week's Sales for each Product
    # Group by Product, Year, and Week of Year to get weekly sales per product
    product_weekly_sales = df_temp.groupby(['Product', 'year', 'week_of_year'])['Total Sales'].sum().reset_index()
    # Shift by 1 period to get the previous week's sales for that product
    product_weekly_sales['Lag_Product_Weekly_Sales'] = product_weekly_sales.groupby('Product')['Total Sales'].shift(1)

    # Merge this lagged data back into the original dataframe based on Product, Year, Week of Year
    df_temp = pd.merge(df_temp, product_weekly_sales[['Product', 'year', 'week_of_year', 'Lag_Product_Weekly_Sales']],
                         on=['Product', 'year', 'week_of_year'],
                         how='left')
    df_temp['Lag_Product_Weekly_Sales'] = df_temp['Lag_Product_Weekly_Sales'].fillna(0) # Fill NaNs for first week's data

    # Sort data for meaningful lag calculations within groups (for category)
    df_temp = df_temp.sort_values(by=['Category', 'Date'])

    # Calculate Last Week's Sales for each Category
    # Group by Category, Year, and Week of Year to get weekly sales per category
    category_weekly_sales = df_temp.groupby(['Category', 'year', 'week_of_year'])['Total Sales'].sum().reset_index()
    # Shift by 1 period to get the previous week's sales for that category
    category_weekly_sales['Lag_Category_Weekly_Sales'] = category_weekly_sales.groupby('Category')['Total Sales'].shift(1)

    # Merge this lagged data back into the original dataframe based on Category, Year, Week of Year
    df_temp = pd.merge(df_temp, category_weekly_sales[['Category', 'year', 'week_of_year', 'Lag_Category_Weekly_Sales']],
                         on=['Category', 'year', 'week_of_year'],
                         how='left')
    df_temp['Lag_Category_Weekly_Sales'] = df_temp['Lag_Category_Weekly_Sales'].fillna(0) # Fill NaNs for first week's data

    # Reset original sort order if desired, or keep sorted for time-series operations later
    df_temp = df_temp.sort_values(by='Date')

    return df_temp

# Apply to train, val, test dataframes
train_df = add_weekly_lag_features(train_df)
val_df = add_weekly_lag_features(val_df)
test_df = add_weekly_lag_features(test_df)

print("Train DataFrame with new lag features (head):\n")
display(train_df[['Date', 'Product', 'Category', 'Total Sales', 'week_of_year', 'Lag_Product_Weekly_Sales', 'Lag_Category_Weekly_Sales']].head())

print("\nValidation DataFrame with new lag features (head):\n")
display(val_df[['Date', 'Product', 'Category', 'Total Sales', 'week_of_year', 'Lag_Product_Weekly_Sales', 'Lag_Category_Weekly_Sales']].head())

print("\nTest DataFrame with new lag features (head):\n")
display(test_df[['Date', 'Product', 'Category', 'Total Sales', 'week_of_year', 'Lag_Product_Weekly_Sales', 'Lag_Category_Weekly_Sales']].head())


### 4.2 Categorical Data Transformation

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Category', 'Payment Method', 'Customer Location']

# Initialize LabelEncoders for each categorical column
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Fit encoder only on the training data to avoid data leakage
    train_df[col + '_encoded'] = le.fit_transform(train_df[col])

    # Transform validation and test data using the fitted encoder
    # Handle new categories that might appear in val/test but not in train
    val_df[col + '_encoded'] = val_df[col].map(lambda s: -1 if s not in le.classes_ else le.transform([s])[0])
    test_df[col + '_encoded'] = test_df[col].map(lambda s: -1 if s not in le.classes_ else le.transform([s])[0])

    label_encoders[col] = le

print("Train DataFrame with encoded categorical features (head):\n")
display(train_df[categorical_cols + [col + '_encoded' for col in categorical_cols]].head())

print("\nValidation DataFrame with encoded categorical features (head):\n")
display(val_df[categorical_cols + [col + '_encoded' for col in categorical_cols]].head())

print("\nTest DataFrame with encoded categorical features (head):\n")
display(test_df[categorical_cols + [col + '_encoded' for col in categorical_cols]].head())


## Stage 5: Scaling and Modeling

In this stage, we will scale the numerical features and train models to predict 'Total Sales'.

In [ ]:
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Feature Selection
# We'll use encoded categories, date features, and lag features.
features = [
    'Price', 'year', 'month', 'day', 'day_of_week', 'week_of_year',
    'is_weekend', 'is_holiday', 'Category_encoded', 'Payment Method_encoded',
    'Customer Location_encoded', 'Lag_Product_Weekly_Sales', 'Lag_Category_Weekly_Sales'
]
target = 'Total Sales'

# 2. Scaling
scaler = StandardScaler()
numerical_cols = ['Price', 'Lag_Product_Weekly_Sales', 'Lag_Category_Weekly_Sales']

# Fit on train, transform all
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
val_df[numerical_cols] = scaler.transform(val_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]
X_test, y_test = test_df[features], test_df[target]

print("Data scaled and features selected.")

In [ ]:
# 3. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_val)
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_preds))
lr_r2 = r2_score(y_val, lr_preds)

print(f"Linear Regression - RMSE: {lr_rmse:.2f}, R2: {lr_r2:.2f}")

In [ ]:
# 4. LightGBM
lgb_model = lgb.LGBMRegressor(verbosity=-1)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])

lgb_preds = lgb_model.predict(X_val)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_preds))
lgb_r2 = r2_score(y_val, lgb_preds)

print(f"LightGBM - RMSE: {lgb_rmse:.2f}, R2: {lgb_r2:.2f}")

## Stage 6: Final Evaluation on Test Set

We now evaluate our trained models on the test set to observe final performance.

In [ ]:
# Final evaluation on Test Set
print("--- Final Test Set Results ---")

# Linear Regression
lr_test_preds = lr_model.predict(X_test)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_preds))
lr_test_r2 = r2_score(y_test, lr_test_preds)
print(f"Linear Regression Test - RMSE: {lr_test_rmse:.2f}, R2: {lr_test_r2:.2f}")

# LightGBM
lgb_test_preds = lgb_model.predict(X_test)
lgb_test_rmse = np.sqrt(mean_squared_error(y_test, lgb_test_preds))
lgb_test_r2 = r2_score(y_test, lgb_test_preds)
print(f"LightGBM Test - RMSE: {lgb_test_rmse:.2f}, R2: {lgb_test_r2:.2f}")

## Stage 6: Final Evaluation on Test Set

We now evaluate our trained models on the test set to observe final performance.

In [ ]:
# Final evaluation on Test Set
print("--- Final Test Set Results ---")

# Linear Regression
lr_test_preds = lr_model.predict(X_test)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_preds))
lr_test_r2 = r2_score(y_test, lr_test_preds)
print(f"Linear Regression Test - RMSE: {lr_test_rmse:.2f}, R2: {lr_test_r2:.2f}")

# LightGBM
lgb_test_preds = lgb_model.predict(X_test)
lgb_test_rmse = np.sqrt(mean_squared_error(y_test, lgb_test_preds))
lgb_test_r2 = r2_score(y_test, lgb_test_preds)
print(f"LightGBM Test - RMSE: {lgb_test_rmse:.2f}, R2: {lgb_test_r2:.2f}")

### Additional Evaluation Metrics: MSE and RMSE

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

print("--- Additional Evaluation Metrics (Test Set) ---")

# Linear Regression
lr_test_mse = mean_squared_error(y_test, lr_test_preds)
lr_test_rmse = np.sqrt(lr_test_mse)
print(f"Linear Regression Test - MSE: {lr_test_mse:.2f}, RMSE: {lr_test_rmse:.2f}")

# LightGBM
lgb_test_mse = mean_squared_error(y_test, lgb_test_preds)
lgb_test_rmse = np.sqrt(lgb_test_mse)
print(f"LightGBM Test - MSE: {lgb_test_mse:.2f}, RMSE: {lgb_test_rmse:.2f}")